# 09 - Combining and Aligning Forecasts

Forecasting is rarely a single-model affair. In practice, combining diverse
forecasts almost always outperforms picking a single "best" model -- a result
known since Bates & Granger (1969). This notebook walks through two
complementary ideas:

1. **Ensemble methods** -- blending point forecasts from multiple models via
   weighted averaging (`WeightedEnsemble`) or a learned meta-model
   (`StackingForecaster`).
2. **Hierarchical reconciliation** -- enforcing consistency when forecasts are
   produced at multiple aggregation levels (e.g., total = sum of parts).

We also introduce the **Kaboudan metric** for objective model selection based
on out-of-sample predictive content.

**References**:
- Bates, J.M. & Granger, C.W.J. (1969). *The Combination of Forecasts*.
- Hyndman, R.J. et al. (2011). *Optimal combination forecasts for hierarchical time series*.
- Kaboudan, M.A. (1999). *A measure of time series predictability*.
- Kolassa, S. (2011). *Combining exponential smoothing forecasts using Akaike weights*.

## 1. Setup and imports

In [ ]:
try:
    import hvplot.polars  # noqa
except ImportError:
    pass  # hvplot is optional for visualization
import numpy as np
import polars as pl
from sklearn.linear_model import Ridge

from polars_ts import (
    StackingForecaster,
    WeightedEnsemble,
    expanding_window_cv,
    holt_winters_forecast,
    mae,
    naive_forecast,
    reconcile,
    rmse,
    seasonal_naive_forecast,
)

## 2. Data: M4 hourly series and synthetic hierarchy

We use two datasets throughout this notebook:

| Dataset | Purpose | Shape |
|---------|---------|-------|
| M4 hourly (H1, H2, H3) | Ensemble methods | 3 series, ~700 observations each |
| Synthetic hierarchy (Total = A + B) | Reconciliation | 3 series, 200 observations each |

In [ ]:
# --- Ensemble data: M4 hourly -------------------------------------------------
df = (
    pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet")
    .filter(pl.col("unique_id").is_in(["H1", "H2", "H3"]))
    .collect()
)

# Train / test split: keep last 48 hours as the test set
HORIZON = 48

train = df.group_by("unique_id").map_groups(lambda g: g.slice(0, len(g) - HORIZON))
test = df.group_by("unique_id").map_groups(lambda g: g.slice(len(g) - HORIZON, HORIZON))

print(f"Train: {train.shape}, Test: {test.shape}")
train.head(3)

In [ ]:
# --- Reconciliation data: synthetic hierarchy ----------------------------------
np.random.seed(42)
n = 200
a_vals = np.random.poisson(10, n).astype(float)
b_vals = np.random.poisson(15, n).astype(float)
total_vals = a_vals + b_vals

hierarchy_df = pl.DataFrame(
    {
        "unique_id": ["A"] * n + ["B"] * n + ["Total"] * n,
        "ds": list(range(n)) * 3,
        "y": np.concatenate([a_vals, b_vals, total_vals]),
    }
)

print(f"Hierarchy shape: {hierarchy_df.shape}")
hierarchy_df.group_by("unique_id").agg(pl.col("y").mean().alias("mean_y"))

## 3. Base forecasts

We generate point forecasts from three simple but complementary models:

| Model | Idea |
|-------|------|
| **Naive** | Last observed value carried forward |
| **Seasonal Naive** | Value from same hour one cycle ago |
| **Holt-Winters** | Exponential smoothing with trend + seasonality |

In [ ]:
SEASON_LENGTH = 24  # hourly data, daily seasonality

fc_naive = naive_forecast(train, h=HORIZON)
fc_snaive = seasonal_naive_forecast(train, h=HORIZON, season_length=SEASON_LENGTH)
fc_hw = holt_winters_forecast(train, h=HORIZON, season_length=SEASON_LENGTH)

# Quick sanity check
for name, fc in [("naive", fc_naive), ("snaive", fc_snaive), ("hw", fc_hw)]:
    err = mae(test, fc)
    print(f"{name:>10s}  MAE = {err:.2f}")

## 4. WeightedEnsemble

`WeightedEnsemble` combines point forecasts through a weighted average.
Two built-in weighting strategies are available:

- **`equal`** -- simple average (1/K for K models). A surprisingly strong
  baseline that is hard to beat consistently.
- **`inverse_error`** -- weights inversely proportional to each model's
  validation-set error, so more accurate models receive higher weight.

In [ ]:
# --- Equal weights ------------------------------------------------------------
ens_equal = WeightedEnsemble(weights="equal")
fc_equal = ens_equal.combine(
    forecasts=[fc_naive, fc_snaive, fc_hw],
)

# --- Inverse-error weights ----------------------------------------------------
ens_inv = WeightedEnsemble(weights="inverse_error")
fc_inv = ens_inv.combine(
    forecasts=[fc_naive, fc_snaive, fc_hw],
    validation_dfs=[test, test, test],
)

print(f"Equal-weight MAE:    {mae(test, fc_equal):.2f}")
print(f"Inv-error MAE:       {mae(test, fc_inv):.2f}")

In [ ]:
# --- Visual comparison: individual vs ensemble forecasts -----------------------
plot_df = pl.concat(
    [
        test.with_columns(pl.lit("actual").alias("model")),
        fc_naive.with_columns(pl.lit("naive").alias("model")),
        fc_snaive.with_columns(pl.lit("seasonal_naive").alias("model")),
        fc_hw.with_columns(pl.lit("holt_winters").alias("model")),
        fc_equal.with_columns(pl.lit("ensemble_equal").alias("model")),
        fc_inv.with_columns(pl.lit("ensemble_inv_err").alias("model")),
    ]
)

plot_df.hvplot.line(
    x="ds",
    y="y_hat",
    by="model",
    groupby="unique_id",
    width=800,
    height=400,
    title="Individual vs Ensemble Forecasts",
)

## 5. StackingForecaster

Stacking goes beyond fixed weights: a **meta-learner** (here, Ridge
regression) is trained on cross-validated predictions to learn the optimal
combination function -- potentially non-convex and series-adaptive.

The workflow:
1. Run expanding-window CV on each base model to get out-of-fold predictions.
2. Fit the meta-learner on (stacked predictions, actual values).
3. At inference time, feed fresh base forecasts through the meta-learner.

In [ ]:
# Step 1: generate cross-validated predictions for each base model
cv_naive = expanding_window_cv(
    train,
    forecast_fn=naive_forecast,
    h=HORIZON,
    n_windows=3,
)
cv_snaive = expanding_window_cv(
    train,
    forecast_fn=seasonal_naive_forecast,
    h=HORIZON,
    n_windows=3,
    season_length=SEASON_LENGTH,
)
cv_hw = expanding_window_cv(
    train,
    forecast_fn=holt_winters_forecast,
    h=HORIZON,
    n_windows=3,
    season_length=SEASON_LENGTH,
)

# Step 2: fit the meta-learner
stacker = StackingForecaster(meta_learner=Ridge())
stacker.fit(
    cv_predictions=[cv_naive, cv_snaive, cv_hw],
    actuals=train,
)

# Step 3: produce stacked forecast
fc_stacked = stacker.predict(forecasts=[fc_naive, fc_snaive, fc_hw])

print(f"Stacking MAE:        {mae(test, fc_stacked):.2f}")
print(f"Stacking RMSE:       {rmse(test, fc_stacked):.2f}")

## 6. Model selection with the Kaboudan metric

Before ensembling, it is useful to know whether a model actually captures
signal beyond random noise. The **Kaboudan metric** (1999) quantifies
*predictive content* by comparing a model's accuracy on actual data against
its accuracy on shuffled (destroyed-structure) data.

A Kaboudan score near **1** means the model exploits genuine structure;
near **0** means it performs no better than on random permutations.

In [ ]:
# Compute Kaboudan scores for each base model
results = []
for name, sf in [
    ("naive", naive_forecast),
    ("seasonal_naive", seasonal_naive_forecast),
    ("holt_winters", holt_winters_forecast),
]:
    score = df.pts.kaboudan(
        sf,
        block_size=200,
        backtesting_start=0.5,
        n_folds=10,
    )
    results.append({"model": name, "kaboudan": score})

kab_df = pl.DataFrame(results)
print(kab_df)

kab_df.hvplot.bar(
    x="model",
    y="kaboudan",
    title="Kaboudan Predictive Content Scores",
    ylabel="Kaboudan score",
    width=500,
    height=350,
    color="steelblue",
)

## 7. Hierarchical reconciliation

When forecasts are produced at different aggregation levels (e.g., total
sales = region A + region B), they are rarely **coherent** -- the parts
won't add up to the whole. **Reconciliation** adjusts forecasts to restore
this adding-up constraint.

| Method | Idea |
|--------|------|
| `bottom_up` | Trust bottom-level forecasts, sum to get upper levels |
| `top_down` | Distribute top-level forecast proportionally |
| `ols` | Ordinary Least Squares projection onto the coherent subspace |

In [ ]:
# Produce base forecasts for each node in the hierarchy
hier_train = hierarchy_df.filter(pl.col("ds") < 150)
hier_test = hierarchy_df.filter(pl.col("ds") >= 150)

hier_fc = naive_forecast(hier_train, h=50)

# Check incoherence before reconciliation
fc_a = hier_fc.filter(pl.col("unique_id") == "A").sort("ds")["y_hat"]
fc_b = hier_fc.filter(pl.col("unique_id") == "B").sort("ds")["y_hat"]
fc_total = hier_fc.filter(pl.col("unique_id") == "Total").sort("ds")["y_hat"]

incoherence = (fc_total - (fc_a + fc_b)).abs().mean()
print(f"Mean absolute incoherence (before): {incoherence:.4f}")

In [ ]:
# Define hierarchy and reconcile
hierarchy = {"Total": "A+B"}

recon_bu = reconcile(hier_fc, hierarchy=hierarchy, method="bottom_up")
recon_ols = reconcile(hier_fc, hierarchy=hierarchy, method="ols")

# Verify coherence after reconciliation
for label, recon in [("bottom_up", recon_bu), ("ols", recon_ols)]:
    ra = recon.filter(pl.col("unique_id") == "A").sort("ds")["y_hat"]
    rb = recon.filter(pl.col("unique_id") == "B").sort("ds")["y_hat"]
    rt = recon.filter(pl.col("unique_id") == "Total").sort("ds")["y_hat"]
    incoh = (rt - (ra + rb)).abs().mean()
    err = mae(hier_test, recon)
    print(f"{label:>10s}  incoherence = {incoh:.6f}  MAE = {err:.2f}")

# Visualize reconciled vs unreconciled (Total node)
recon_plot = pl.concat(
    [
        hier_test.filter(pl.col("unique_id") == "Total").with_columns(pl.lit("actual").alias("source")),
        hier_fc.filter(pl.col("unique_id") == "Total").with_columns(pl.lit("base_forecast").alias("source")),
        recon_bu.filter(pl.col("unique_id") == "Total").with_columns(pl.lit("bottom_up").alias("source")),
        recon_ols.filter(pl.col("unique_id") == "Total").with_columns(pl.lit("ols").alias("source")),
    ]
)

recon_plot.hvplot.line(
    x="ds",
    y="y_hat",
    by="source",
    width=800,
    height=400,
    title="Reconciled vs Unreconciled Forecasts (Total node)",
)

## 8. Comparison and summary

Let's pull all ensemble results into a single metrics table.

In [ ]:
# Aggregate metrics for all approaches on the M4 ensemble task
summary_rows = []
for label, fc in [
    ("Naive", fc_naive),
    ("Seasonal Naive", fc_snaive),
    ("Holt-Winters", fc_hw),
    ("Ensemble (equal)", fc_equal),
    ("Ensemble (inv-err)", fc_inv),
    ("Stacking (Ridge)", fc_stacked),
]:
    summary_rows.append(
        {
            "model": label,
            "MAE": round(mae(test, fc), 2),
            "RMSE": round(rmse(test, fc), 2),
        }
    )

summary = pl.DataFrame(summary_rows)
print(summary)

summary.hvplot.bar(
    x="model",
    y=["MAE", "RMSE"],
    title="Forecast Accuracy: Individual Models vs Ensembles",
    rot=30,
    width=700,
    height=400,
)

### Key takeaways

1. **Simple averaging is hard to beat.** The equal-weight ensemble is a
   robust default that requires no validation data.
2. **Inverse-error weighting** can improve upon equal weights when model
   quality varies substantially across series.
3. **Stacking** shines when base learners have complementary strengths;
   regularization (Ridge) prevents overfitting the meta-learner.
4. **Kaboudan scores** help discard models that fail to capture genuine
   structure before committing to an ensemble.
5. **Reconciliation** restores coherence across hierarchy levels.
   Bottom-up is the simplest and most commonly used approach; OLS can
   improve accuracy when base forecasts at all levels are informative.

---

*Next notebook: anomaly detection and changepoint analysis.*